In [ ]:
import subprocess
from dotenv import load_dotenv
load_dotenv(override=True) 

True

<h2>Download clip 25s from youtube videos</h2>

In [92]:
PROJECT_FOLDER = '/Users/sangdo/Downloads/movie-top-list/'

In [93]:
import json

def download_1_trailer(trailer_id, folder_path, start_second, length=25):
    url = "https://www.youtube.com/watch?v=" + trailer_id
    final_output = folder_path + "/" + trailer_id + ".mp4"
    end_second = start_second + length
    start_ts = f"00:{start_second // 60:02d}:{start_second % 60:02d}"
    end_ts = f"00:{end_second // 60:02d}:{end_second % 60:02d}"

    # Check available formats before downloading
    result = subprocess.run(
        ["yt-dlp", "-j", "--quiet", "--no-warnings", url],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'Error fetching video info for {trailer_id}:')
        print(result.stderr)
        return
    info = json.loads(result.stdout)
    resolutions = sorted(set(
        f"{f.get('width')}x{f.get('height')}"
        for f in info.get('formats', [])
        if f.get('width') and f.get('height')
    ))
    print(f"Available resolutions for {trailer_id}: {', '.join(resolutions)}")
    has_1280 = any(f.get('width') == 1280 and f.get('height') == 720 for f in info.get('formats', []))
    if not has_1280:
        print(f'Skipped {trailer_id}: 1280x720 format not available')
        return

    # Download 1280x720 video + best audio
    cmd_dl = [
        "yt-dlp",
        "--quiet",
        "--no-warnings",
        "--force-overwrites",
        "--download-sections", f"*{start_ts}-{end_ts}",
        "--force-keyframes-at-cuts",
        "-f", "bv*[width=1280][height=720]+ba",
        "--merge-output-format", "mp4",
        "-o", final_output,
        url
    ]
    subprocess.run(cmd_dl, check=True)
    print('Finish downloading: ' + trailer_id)

def create_thumbnail(trailer_id, folder_path, at_second):
    video_path = folder_path + "/" + trailer_id + ".mp4"
    thumb_path = folder_path + "/" + trailer_id + "_thumb.png"
    cmd = [
        "ffmpeg", "-y",
        "-loglevel", "quiet",
        "-ss", str(at_second),
        "-i", video_path,
        "-frames:v", "1",
        "-update", "1",
        "-vf", "scale=250:150",
        thumb_path
    ]
    subprocess.run(cmd, check=True)
    print('Thumbnail saved: ' + thumb_path)
    return thumb_path

In [95]:
#test
category = 'tom_cruise'
folder_path = PROJECT_FOLDER + 'clips/' + category + '/'
# download_1_trailer('bWWMMyDtMoQ', folder_path, 20)  #around 36s to download
# create_thumbnail('bWWMMyDtMoQ', folder_path, 5)